In [ ]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

from common_module.model_hub.llm_factory import ChatOpenAI, LLMRegistry
from common_module.parse_mill.str_refinery import StrOutputParser, StrParser


class LLMRunner:
    def __init__(
        self,
        llm: ChatOpenAI | None = None,
        parser: StrOutputParser | None = None,
    ) -> None:
        self.llm = llm or LLMRegistry.get_default()
        self.parser = parser or StrParser.get()

    def invoke(self, prompt: str | list[BaseMessage]) -> str:
        response = self.llm.invoke(prompt)
        return self.parser.invoke(input=response)

    async def ainvoke(self, prompt: str | list[BaseMessage]) -> str:
        response = await self.llm.ainvoke(prompt)
        return await self.parser.ainvoke(input=response)

    async def abatch(
        self, prompts: list[str | list[BaseMessage]]
    ) -> list[str | Exception]:
        responses: list[AIMessage | Exception] = await self.llm.abatch(
            prompts, return_exceptions=True
        )

        results: list[str | Exception] = [
            r if isinstance(r, Exception) else "" for r in responses  # temp
        ]

        ok_indexes: list[int] = [
            i for i, r in enumerate(responses) if not isinstance(r, Exception)
        ]
        ok_inputs: list[AIMessage] = [responses[i] for i in ok_indexes]

        if ok_inputs:
            parsed: list[str | Exception] = await self.parser.abatch(
                ok_inputs, return_exceptions=True
            )
            for i, item in zip(ok_indexes, parsed):
                results[i] = item

        return results

    async def structured_ainvoke(
        self, prompt: str, schema: type[BaseModel]
    ) -> BaseModel:
        structured_llm = self.llm.with_structured_output(schema=schema)

        try:
            response = await structured_llm.ainvoke(input=prompt)
        except Exception:
            parser = PydanticOutputParser(pydantic_object=schema)
            fixing_parser = OutputFixingParser.from_llm(llm=self.llm, parser=parser)
            fallback_prompt = f"{prompt}\n\n{parser.get_format_instructions()}"
            raw_response = await self.llm.ainvoke(input=fallback_prompt)
            response = await fixing_parser.ainvoke(input=raw_response)

        return response

    async def structured_abatch(
        self,
        prompts: list[str],
        schema: type[BaseModel],
    ) -> list[BaseModel | Exception]:
        structured_llm = self.llm.with_structured_output(schema=schema)

        try:
            results: list[BaseModel | Exception] = await structured_llm.abatch(
                prompts, return_exceptions=True
            )
        except Exception:
            # whole structured path failed → fallback for every prompt
            return await self._structured_fallback_abatch(prompts, schema)

        fail_indexes: list[int] = [
            i for i, r in enumerate(results) if isinstance(r, Exception)
        ]
        if not fail_indexes:
            return results

        # only retry the failed items
        fallback: list[BaseModel | Exception] = await self._structured_fallback_abatch(
            [prompts[i] for i in fail_indexes], schema
        )
        for i, item in zip(fail_indexes, fallback):
            results[i] = item

        return results

    async def _structured_fallback_abatch(
        self,
        prompts: list[str],
        schema: type[BaseModel],
    ) -> list[BaseModel | Exception]:
        parser: PydanticOutputParser[BaseModel] = PydanticOutputParser(
            pydantic_object=schema
        )
        fixing_parser: object = OutputFixingParser.from_llm(llm=self.llm, parser=parser)
        format_instructions: str = parser.get_format_instructions()

        fallback_prompts: list[str] = [f"{p}\n\n{format_instructions}" for p in prompts]
        raw_responses: list[AIMessage | Exception] = await self.llm.abatch(
            fallback_prompts, return_exceptions=True
        )

        results: list[BaseModel | Exception] = [
            r if isinstance(r, Exception) else None  # type: ignore[list-item]
            for r in raw_responses
        ]

        ok_indexes: list[int] = [
            i for i, r in enumerate(raw_responses) if not isinstance(r, Exception)
        ]
        ok_inputs: list[AIMessage | Exception] = [raw_responses[i] for i in ok_indexes]

        if ok_inputs:
            parsed: list[BaseModel | Exception] = await fixing_parser.abatch(
                ok_inputs, return_exceptions=True
            )
            for i, item in zip(ok_indexes, parsed):
                results[i] = item

        return results

In [2]:
lp = LLMProcessing()

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


In [9]:
lp.invoke([HumanMessage(content="hi")])

'Hi there! How can I help today? If you’d like, tell me what you’re looking for—information, ideas, writing help, tech tips, or anything else.'

In [6]:
from langchain_core.messages import HumanMessage

In [8]:
await lp.ainvoke([HumanMessage(content="hi")])

'Hi there! How can I help today? Do you want to chat, get information, solve a problem, or need help with something specific?'

In [14]:
ans = await lp.abatch([[HumanMessage(content='hi')], [HumanMessage(content='who is pm of nepal')]])
ans

['Hi there! How can I help today? If you’re not sure, tell me your goal or what you’re working on, and I’ll tailor my help. For example:\n- Answer a question or explain a concept\n- Help with writing or editing\n- Summarize or translate\n- Debug code or brainstorm ideas\n- Plan a task or project\n\nWhat would you like to do?',
 'Do you want the current prime minister right now? I can provide the latest, but my internal knowledge may be outdated.\n\nAs of my last update (June 2024), the Prime Minister of Nepal is Pushpa Kamal Dahal, commonly known as Prachanda, who has been serving since December 2022.\n\nWould you like me to check the most current status for today?']

In [21]:
from pydantic import BaseModel


class Person(BaseModel):
    name: str
    age: int

queries = ["who is PM of india", "Bhaijaan of bollywood"]
await lp.structured_ainvoke(queries, Person)

Person(name='Narendra Modi', age=70)

In [26]:
await lp.structured_abatch(queries, Person)

[Person(name='Narendra Modi', age=69), Person(name='Salman Khan', age=59)]